In [ ]:
import pandas as pd

import gc
import os
import torch

from text_classification import (
    TextClassifierTrainConfig,
    train_model,
    plot_history,
    print_validation_reports
)

c:\Users\borqu\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda
Device Name: NVIDIA GeForce RTX 3060 Laptop GPU


In [2]:
# 1) borra referencias grandes que sigan vivas
for name in [
    "model", "optimizer", "scheduler",
    "train_loader", "val_loader",
    "train_ds", "val_ds",
    "batch", "outputs", "loss", "logits", "preds"
]:
    if name in globals():
        del globals()[name]

# 2) limpia RAM de Python
gc.collect()

# 3) intenta sincronizar y vaciar caché CUDA
if torch.cuda.is_available():
    try:
        torch.cuda.synchronize()
    except Exception:
        pass

    try:
        torch.cuda.empty_cache()
    except Exception as e:
        print("empty_cache falló:", repr(e))

    try:
        torch.cuda.ipc_collect()
    except Exception as e:
        print("ipc_collect falló:", repr(e))

# Experimento de entrenamiento X 

La siguiente celda muestra un ejemplo de uso
Es posible variar múltiples hiperparámetros del entrenamiento de manera simple (lr, wd, freeze strategy, etc.) según los parámetros entregados a TextClassifierTrainConfig. 

Es posible usar otra arquitectura distinta de Modernbert-base. Aunque se puede experimentar con modernbert large, llama, berta, etc. variando model_name, esto solamente ha sido testeado para Modernbert-base y ModernBERT-large.

In [ ]:
EXPERIMENT_NAME = "nombre_experimento"

config = TextClassifierTrainConfig(
    model_name="answerdotai/ModernBERT-large",
    text_col="comentario_en",
    label_col="categoria",
    valid_labels = (1, 2, 3, 4, 5, 6, 7, 8, 9),
    other_class_code = 9,
    class_names = {1: 'Comunicación y Presencia',
                   2: 'Salud',
                   3: 'Asesoría preventiva',
                   4: 'Capacitaciones',
                   5: 'Plataforma',
                   6: 'SEL',
                   7: 'Calificación',
                   8: 'Requerimiento',
                   9: 'Otros'},
    max_length=128,
    train_batch_size=16,
    val_batch_size=32,
    learning_rate=1e-5,
    num_epochs=100,
    use_class_weights=True,
    weight_decay=0.2,
    model_save_name=f"{EXPERIMENT_NAME}.pt",
    freeze_strategy="all_but_top_n",
    unfreeze_top_n=6,
    freeze_embeddings=True,
    use_early_stopping = True,
    early_stopping_patience = 15,
    embedding_dropout = 0.25,
    attention_dropout = 0.25,
    mlp_dropout = 0.28,
    classifier_dropout = 0.35,
)

# 2) train_df  y val_df se obtienen de archivos ya guardados
train_df = pd.concat([pd.read_excel(os.path.join('path', 'to', 'train_base_df.xlsx')),
                      pd.read_excel(os.path.join('path', 'to', 'back_translate_data_aug_df.xlsx')),
                      pd.read_excel(os.path.join('path', 'to', 'synonym_replacement_data_aug_df.xlsx')),
                      pd.read_excel(os.path.join('path', 'to', 'ctx_ins_repl_data_aug_df.xlsx'))
                      ]).sample(frac=1)
train_df.drop_duplicates(subset='comentario_en', inplace=True)
                     
val_df = pd.concat([pd.read_excel(os.path.join('path', 'to', 'val_base_df.xlsx'))])

print("\nDistribución TRAIN:")
print(train_df[config.label_col].value_counts(normalize=True).sort_index())

print("\nDistribución VAL:")
print(val_df[config.label_col].value_counts(normalize=True).sort_index())

# 3) entrenar
model, tokenizer, history_df = train_model(train_df, val_df, config)

# 4) ver evolución
display(history_df)
plot_history(history_df)

# 5) reportes detallados en validación
cm_df = print_validation_reports(model, val_df, tokenizer, config)
display(cm_df)